In [1]:
# Instalación de dependencias (solo necesario en Colab)
!pip install yfinance scikit-learn ipywidgets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 14.1 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN


# ── Funciones ────────────────────────────────────────────────────────────────

def descargar_datos(ticker):
    df = yf.download(
        tickers=ticker,
        period='max',
        group_by='ticker',
        progress=False,
        auto_adjust=False
    )
    df = df.stack(level=0).rename_axis(['Date', 'Stock']).reset_index()
    df = df.rename(columns={
        'Open': '1. Open', 'High': '2. High', 'Low': '3. Low',
        'Close': '4. Close', 'Volume': '5. Volume',
        'Adj Close': '7. Adj Close', 'Stock': '6. Stock'
    })
    df = df.drop(columns=['1. Open', '2. High', '3. Low', '4. Close', '5. Volume'])
    return df


def calcular_emas(data):
    data = data.copy()
    data['EMA_20'] = data['7. Adj Close'].ewm(span=20, adjust=False).mean()
    data['EMA_50'] = data['7. Adj Close'].ewm(span=50, adjust=False).mean()
    return data


def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)


def entrenar_y_predecir(df_stocks, ticker, fecha_inicio, seq_length=7, dias=7, epochs=50):
    data = df_stocks[df_stocks['6. Stock'] == ticker].copy()
    data.sort_values('Date', inplace=True)
    data.reset_index(drop=True, inplace=True)
    data = calcular_emas(data)

    data_rnn = data[data['Date'] >= pd.Timestamp(fecha_inicio)].copy()
    data_rnn.sort_values('Date', ascending=True, inplace=True)
    data_rnn.reset_index(drop=True, inplace=True)

    if len(data_rnn) < seq_length + 5:
        raise ValueError(f'Muy pocos datos desde {fecha_inicio}. Elegi una fecha mas antigua.')

    scaler = MinMaxScaler()
    data_rnn['adj_close_scaled'] = scaler.fit_transform(data_rnn[['7. Adj Close']])

    X, y = create_sequences(data_rnn['adj_close_scaled'].values, seq_length)
    split = int(0.8 * len(X))
    X_train, y_train = X[:split], y[:split]
    X_train = X_train.reshape(X_train.shape[0], seq_length, 1)

    model = Sequential([
        SimpleRNN(256, input_shape=(seq_length, 1)),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    model.fit(X_train, y_train, epochs=epochs, batch_size=64,
              validation_split=0.1, verbose=0)

    last_seq = data_rnn['adj_close_scaled'].values[-seq_length:].copy()
    future_preds = []
    for _ in range(dias):
        pred = model.predict(last_seq.reshape(1, seq_length, 1), verbose=0)[0, 0]
        future_preds.append(pred)
        last_seq = np.append(last_seq[1:], pred)

    predictions = scaler.inverse_transform(
        np.array(future_preds).reshape(-1, 1)
    ).flatten()

    last_date = data_rnn['Date'].iloc[-1]
    next_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=dias)
    df_future = pd.DataFrame({'Date': next_dates, '7. Adj Close': predictions})

    return data, data_rnn, df_future


def graficar(data, data_rnn, df_future, ticker):
    precio_actual = data['7. Adj Close'].iloc[-1]
    ema20_actual  = data['EMA_20'].iloc[-1]
    ema50_actual  = data['EMA_50'].iloc[-1]

    if precio_actual > ema20_actual and precio_actual > ema50_actual:
        senal = 'Tendencia ALCISTA: precio por encima de EMA 20 y EMA 50'
    elif precio_actual < ema20_actual and precio_actual < ema50_actual:
        senal = 'Tendencia BAJISTA: precio por debajo de EMA 20 y EMA 50'
    else:
        senal = 'Tendencia MIXTA: precio entre EMA 20 y EMA 50'

    print(f'Precio actual: ${precio_actual:.2f} | EMA 20: ${ema20_actual:.2f} | EMA 50: ${ema50_actual:.2f}')
    print(f'Senal: {senal}\n')

    fig1, ax1 = plt.subplots(figsize=(14, 4))
    ax1.plot(data['Date'], data['7. Adj Close'],
             label='Precio Ajustado (Historico)', color='#1f77b4', linewidth=1.2)
    ax1.set_title(f'Historial del Precio Ajustado de {ticker}', fontsize=13)
    ax1.set_xlabel('Fecha')
    ax1.set_ylabel('Precio (USD)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    fig1.tight_layout()
    plt.show()

    fig2, ax2 = plt.subplots(figsize=(14, 4))
    ultimos = data_rnn[['Date', '7. Adj Close', 'EMA_20', 'EMA_50']].tail(30)
    ax2.plot(ultimos['Date'], ultimos['7. Adj Close'],
             label='Ultimos 30 dias reales', color='#1f77b4', linewidth=1.2)
    ax2.plot(ultimos['Date'], ultimos['EMA_20'],
             label='EMA 20', color='#ff7f0e', linewidth=1.2, linestyle='--')
    ax2.plot(ultimos['Date'], ultimos['EMA_50'],
             label='EMA 50', color='#2ca02c', linewidth=1.2, linestyle='--')
    ax2.plot(df_future['Date'], df_future['7. Adj Close'],
             label=f'Prediccion RNN ({len(df_future)} dias)',
             color='#d62728', linewidth=2, linestyle='--', marker='o', markersize=5)
    ax2.axvline(x=data_rnn['Date'].iloc[-1], color='gray', linestyle=':', linewidth=1,
                label='Hoy')
    ax2.set_title(f'Prediccion RNN para {ticker} (proximos {len(df_future)} dias)', fontsize=13)
    ax2.set_xlabel('Fecha')
    ax2.set_ylabel('Precio (USD)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    fig2.tight_layout()
    plt.show()


# ── Tickers de referencia ────────────────────────────────────────────────────

TICKERS_REF = {
    # Tecnologia
    'AAPL': 'Apple', 'MSFT': 'Microsoft', 'GOOGL': 'Alphabet', 'AMZN': 'Amazon',
    'META': 'Meta', 'TSLA': 'Tesla', 'NVDA': 'NVIDIA', 'NFLX': 'Netflix',
    'ADBE': 'Adobe', 'CRM': 'Salesforce', 'ORCL': 'Oracle', 'IBM': 'IBM',
    'INTC': 'Intel', 'AMD': 'AMD', 'CSCO': 'Cisco',
    # Finanzas
    'JPM': 'JPMorgan', 'BAC': 'Bank of America', 'GS': 'Goldman Sachs',
    'V': 'Visa', 'MA': 'Mastercard', 'BRK-B': 'Berkshire B', 'PYPL': 'PayPal',
    # Salud
    'JNJ': 'Johnson & Johnson', 'PFE': 'Pfizer', 'LLY': 'Eli Lilly', 'UNH': 'UnitedHealth',
    # Consumo / Retail
    'WMT': 'Walmart', 'MCD': "McDonald's", 'KO': 'Coca-Cola', 'DIS': 'Disney',
    'NKE': 'Nike', 'SBUX': 'Starbucks', 'COST': 'Costco',
    # Energia
    'XOM': 'ExxonMobil', 'CVX': 'Chevron',
    # Otros
    'BABA': 'Alibaba', 'TSM': 'TSMC', 'SPOT': 'Spotify', 'ABNB': 'Airbnb',
}

# ── Widgets ──────────────────────────────────────────────────────────────────

ancho = '180px'

ticker_input = widgets.Text(
    value='',
    placeholder='Ej: AAPL, TSLA...',
    layout=widgets.Layout(width=ancho, height='32px')
)

sugerencias = widgets.Select(
    options=[],
    rows=5,
    layout=widgets.Layout(width=ancho, display='none')
)

fecha_input = widgets.DatePicker(
    value=pd.Timestamp('2024-01-01').date(),
    layout=widgets.Layout(width=ancho)
)

boton = widgets.Button(
    description='Establecer prediccion',
    button_style='primary',
    layout=widgets.Layout(width=ancho, height='32px')
)

boton.style.button_color = '#1976D2'
boton.add_class('boton-redondeado')

display(widgets.HTML("""
<style>
  .boton-redondeado button {
    border-radius: 20px !important;
    text-align: center !important;
    font-size: 13px !important;
    font-family: sans-serif !important;
  }
  .campo-label {
    font-family: sans-serif;
    font-size: 13px;
    font-weight: bold;
    margin: 0 0 4px 0;
    color: #222;
  }
  .widget-text input, .widget-datepicker input {
    border-radius: 20px !important;
    padding: 0 12px !important;
    border: 1px solid #bbb !important;
    font-size: 13px !important;
    height: 32px !important;
  }
  .widget-select select {
    border-radius: 10px !important;
    font-size: 13px !important;
    padding: 4px 8px !important;
  }
  .widget-datepicker {
    overflow: hidden !important;
    height: 32px !important;
  }
  .widget-datepicker input[type=date] {
    overflow: hidden !important;
    scrollbar-width: none !important;
    height: 32px !important;
    box-sizing: border-box !important;
  }
  .widget-datepicker input[type=date]::-webkit-scrollbar {
    display: none !important;
  }
</style>
"""))

def on_ticker_change(change):
    texto = change['new'].strip().upper()
    if len(texto) == 0:
        sugerencias.options = []
        sugerencias.layout.display = 'none'
        return
    coincidencias = [
        f'{k} — {v}' for k, v in TICKERS_REF.items()
        if k.startswith(texto) or v.upper().startswith(texto)
    ]
    if coincidencias:
        sugerencias.options = [''] + coincidencias
        sugerencias.value = ''
        sugerencias.layout.display = ''
    else:
        sugerencias.options = []
        sugerencias.layout.display = 'none'

def on_sugerencia_click(change):
    if change['new'] and change['new'] != '':
        ticker_seleccionado = change['new'].split(' — ')[0]
        # Desconectamos el observer temporalmente para no disparar on_ticker_change
        ticker_input.unobserve(on_ticker_change, names='value')
        ticker_input.value = ticker_seleccionado
        ticker_input.observe(on_ticker_change, names='value')
        sugerencias.options = []
        sugerencias.layout.display = 'none'

ticker_input.observe(on_ticker_change, names='value')
sugerencias.observe(on_sugerencia_click, names='value')

output = widgets.Output()


def on_click(b):
    with output:
        clear_output(wait=True)
        ticker = ticker_input.value.strip().upper()
        fecha = fecha_input.value

        if not ticker:
            print('Ingresa un ticker valido.')
            return

        print(f'Descargando datos de {ticker}...')
        try:
            df_stocks = descargar_datos(ticker)
        except Exception as e:
            print(f'Error al descargar datos: {e}')
            return

        # Validar que el ticker existe y tiene datos
        if df_stocks.empty or ticker not in df_stocks['6. Stock'].values:
            print(f'Lo siento, ticker "{ticker}" no encontrado. Verifica que sea un simbolo valido (ej: AAPL, TSLA, GOOGL).')
            return

        print('Entrenando modelo RNN...')
        try:
            data, data_rnn, df_future = entrenar_y_predecir(df_stocks, ticker, fecha)
        except ValueError as e:
            print(f'Error: {e}')
            return

        print('Listo!\n')
        graficar(data, data_rnn, df_future, ticker)


boton.on_click(on_click)

# ── Layout ────────────────────────────────────────────────────────────────────

titulo = widgets.HTML(
    '<h2 style="font-family:sans-serif; margin:0 0 16px 0;">Prediccion Precio de Acciones</h2>'
)

col_ticker = widgets.VBox([
    widgets.HTML('<p class="campo-label" style="color: white; font-weight: bold; margin-bottom: 5px;">Ticker</p>'),
    ticker_input,
    sugerencias
], layout=widgets.Layout(margin='0 12px 0 0'))

col_fecha = widgets.VBox([
    widgets.HTML('<p class="campo-label" style="color: white; font-weight: bold; margin-bottom: 5px;">Inicio RNN</p>'),
    fecha_input
], layout=widgets.Layout(margin='0 12px 0 0'))

col_boton = widgets.VBox([
    widgets.HTML('<p class="campo-label">&nbsp;</p>'),
    boton
], layout=widgets.Layout(margin='0'))

fila = widgets.HBox(
    [col_ticker, col_fecha, col_boton],
    layout=widgets.Layout(align_items='flex-start')
)

panel = widgets.VBox(
    [titulo, fila],
    layout=widgets.Layout(
        padding='20px',
        border='1px solid #e0e0e0',
        border_radius='10px',
        width='fit-content'
    )
)

display(panel, output)








HTML(value='\n<style>\n  .boton-redondeado button {\n    border-radius: 20px !important;\n    text-align: cent…

Output()